# Does the Incumbent Usually Win the Recompete?

Measures how often the same vendor keeps a contract when the government buys the
same thing again, broken out per agency, using USASpending prime contract data.

Source: USASpending Award Data Archive (public, no signup, no API key).
Range: FY2013-FY2025. Agencies: Navy, Army, Air Force, DHS, VA, GSA, HHS.

Run top to bottom. Cell 2 downloads the full-year archives for five agencies across
thirteen years and writes a slimmed parquet cache,
so a disconnect only costs the file that was in flight.

In [ ]:
import io
import json
import os
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Download DoD once, then split it into Navy / Army / Air Force by sub-agency.
# There are no separate archive files for the military branches.
AGENCIES = {126: "DoD", 63: "DHS", 37: "VA", 40: "GSA", 68: "HHS"}
DOD_SUBS = {
    "Department of the Navy": "Navy",
    "Department of the Army": "Army",
    "Department of the Air Force": "Air Force",
}
FISCAL_YEARS = range(2013, 2026)   # 13 years, so 5-year contracts show two full cycles
FY_LABEL = f"FY{min(FISCAL_YEARS)}-FY{max(FISCAL_YEARS)}"   # every caption reads this
# Set GOVCON_FY to restrict a run to a single fiscal year; unset (the default here)
# downloads the full range in one pass.
if os.environ.get("GOVCON_FY"):
    FISCAL_YEARS = [int(os.environ["GOVCON_FY"])]

MIN_VALUE = 250_000    # roughly the simplified acquisition threshold
EVENT_GAP_D = 180      # awards closer together than this are one award event
LINK_WINDOW_D = 365    # a successor award lands within a year of the old end date
MIN_N = 200            # fewer recompetes than this is too noisy to publish

BLUE, PALE, INK, MUTED, GRID = "#2a78d6", "#9ec5f4", "#0b0b0b", "#52514e", "#e6e5e1"
SURFACE = "#fcfcfb"
# NAICS alongside PSC because a product code alone is too broad to mean "the same
# work": HHS product code R604 holds both a $1.2M mailroom contract and $8.1B of
# COVID vaccine distribution. NAICS separates them.
KEY = ["agency", "awarding_office_code", "product_or_service_code", "naics_code"]
VALUE_RATIO_MAX = 10   # a replacement is a similar size to what it replaces
KEY_BRIEF = ["agency", "product_or_service_code"]   # coarsest grouping: agency + PSC only


try:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/govcon_recompete")
    TMP = Path("/content/tmp")
except ImportError:
    BASE = Path.cwd()
    TMP = Path(os.environ.get("GOVCON_TMP", BASE / "tmp"))

SLIM = BASE / "slim_v2"   # cached parquet per agency-year: keeps IDVs and orders, and the follow-on flag
OUT = BASE / "outputs"
for d in (SLIM, OUT, TMP):
    d.mkdir(parents=True, exist_ok=True)
print("cache:", SLIM, "| scratch:", TMP)

## Download and slim

Archive URLs are not stable strings - the date suffix changes every time
USASpending regenerates the monthly files - so we ask the API for the current URL
rather than hardcoding it. Note `agency` is the toptier_agency_id, not the
3-digit CGAC code.

In [ ]:
import time

LIST_URL = "https://api.usaspending.gov/api/v2/bulk_download/list_monthly_files/"


def retry(fn, attempts=7, base_delay=15, cap=600):
    """Retry with exponential backoff, capped high because the rate limit is not brief.

    The archive server refuses connections for minutes at a time under sustained load,
    so the backoff needs a cap long enough to outlast that window rather than retry
    inside it.
    """
    for i in range(attempts):
        try:
            return fn()
        except (requests.RequestException, OSError) as e:
            if i == attempts - 1:
                raise
            wait = min(base_delay * 2 ** i, cap)
            print(f"    {type(e).__name__}, retrying in {wait}s ({i + 1}/{attempts - 1})", flush=True)
            time.sleep(wait)


def archive_url(agency_id, fy):
    def go():
        r = requests.post(
            LIST_URL, json={"agency": agency_id, "fiscal_year": fy, "type": "contracts"}, timeout=120
        )
        r.raise_for_status()
        return r.json()["monthly_files"]

    files = retry(go)
    full = [f for f in files if f["fiscal_year"] == fy and "_Full_" in f["file_name"]]
    if not full:
        raise RuntimeError(f"no Full file for agency {agency_id} FY{fy}")
    return full[0]["url"], full[0]["file_name"]


def download(url, dest):
    def go():
        with requests.get(url, stream=True, timeout=(30, 300)) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for block in r.iter_content(1 << 20):
                    fh.write(block)
        # A truncated download still leaves a file on disk. Opening the zip's central
        # directory (stored at the end of the archive) is a cheap check that the file is
        # complete, without decompressing its contents.
        with zipfile.ZipFile(dest) as zf:
            if not zf.namelist():
                raise OSError("empty zip")
        return dest

    try:
        return retry(go)
    except Exception:
        dest.unlink(missing_ok=True)
        raise


def original_awards(chunk):
    """Keep every original award, whatever kind of contract it is.

    Rows in this data are transactions, not contracts, and roughly 87% of them are
    modifications to something that already exists - counting those as awards would
    count every edit as a new deal, so that is the only filter here.

    Standalone contracts, IDV vehicles, and orders placed under a vehicle are all kept
    and separated later, at analysis time, so the same cache serves any question about
    contract type without re-downloading anything.
    """
    mod = chunk["modification_number"].fillna("").str.strip()
    return chunk[mod.str.fullmatch(r"0+")]


def slim_one(agency_id, label, fy):
    """Download one agency-year, keep the columns and rows we need, cache as parquet."""
    dest = SLIM / f"{label}_{fy}.parquet"
    if dest.exists():
        return dest

    url, name = archive_url(agency_id, fy)
    zip_path = TMP / name
    print(f"  downloading {name} ...", flush=True)
    download(url, zip_path)

    frames, raw_rows = [], 0
    try:
        with zipfile.ZipFile(zip_path) as zf:
            members = [n for n in zf.namelist() if n.lower().endswith(".csv")]
            for member in members:
                with zf.open(member) as fh:
                    header = pd.read_csv(fh, nrows=0, encoding_errors="replace")
                cols = [c for c in KEEP_COLS if c in header.columns]
                missing = set(KEEP_COLS) - set(cols)
                if missing:
                    print(f"    note: {member} lacks {sorted(missing)}")
                with zf.open(member) as fh:
                    for chunk in pd.read_csv(
                        fh, usecols=cols, dtype=str, chunksize=250_000,
                        encoding_errors="replace", low_memory=False,
                    ):
                        raw_rows += len(chunk)
                        frames.append(original_awards(chunk))
    finally:
        zip_path.unlink(missing_ok=True)

    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=KEEP_COLS)
    df["source_fy"] = fy
    df.to_parquet(dest, index=False)
    share = len(df) / raw_rows if raw_rows else 0
    print(f"  {label} FY{fy}: {raw_rows:,} transactions -> {len(df):,} base awards ({share:.1%})")
    return dest

## Look at one file before deciding what to keep

The archive ships 297 columns per contract, more than any one analysis needs. This
pulls the smallest file in the set and checks what a row represents, which columns
hold what the analysis needs, and where the pitfalls in this specific dataset are -
the assumptions the rest of the notebook relies on.

In [ ]:
_url, _name = archive_url(68, 2025)          # HHS, the smallest file in the set
_probe = TMP / _name
if not _probe.exists():
    download(_url, _probe)

with zipfile.ZipFile(_probe) as _z:
    _member = [n for n in _z.namelist() if n.lower().endswith(".csv")][0]
    peek = pd.read_csv(_z.open(_member), dtype=str, nrows=20_000, encoding_errors="replace")
_probe.unlink(missing_ok=True)

print(f"{peek.shape[1]} columns, sampled {len(peek):,} rows from {_name}\n")
(OUT / "all_columns.txt").write_text("\n".join(peek.columns))

THEMES = {
    "who won it":      r"recipient",
    "what was bought": r"product_or_service|naics",
    "who bought it":   r"awarding_(agency|sub_agency|office)",
    "how much":        r"obligation|value_of_award",
    "when":            r"action_date$|period_of_performance",
    "what kind":       r"award_type|idv_flag|pricing|extent_competed",
    "which contract":  r"piid|modification_number",
}
for label, pat in THEMES.items():
    print(f"{label:18s} {', '.join(c for c in peek.columns if re.search(pat, c))[:150]}")

In [ ]:
# Check 1: for each code/description pair, which column holds which. The _code
# suffix is not a reliable guide - two of these four pairs are swapped and two are
# not, so the value has to be read rather than assumed.
print("Check 1 - which column holds the code and which holds the description:")
for c in ["type_of_contract_pricing", "extent_competed", "award_type",
          "other_than_full_and_open_competition"]:
    print(f"  {c + '_code':32s} -> {peek[c + '_code'].dropna().iloc[0]!r}")
    print(f"  {c:32s} -> {peek[c].dropna().iloc[0]!r}")

# Check 2: a row is a transaction, not a contract. Most rows edit a contract that
# already exists, so counting rows as awards would count every edit as a new deal.
base_share = peek.modification_number.str.strip().str.fullmatch(r"0+").mean()
print(f"\nCheck 2 - {base_share:.1%} of rows are original awards; "
      f"the rest are modifications")

# Check 3: orders placed against a vehicle someone already holds are not
# competitions - they go to that holder by construction.
has_parent = peek.parent_award_id_piid.fillna("").str.strip().ne("").mean()
print(f"Check 3 - {has_parent:.1%} of rows are orders under an existing vehicle")

# Check 4: UEI is backfilled into old records; DUNS was retired and is not.
print(f"\nCheck 4 - vendor id coverage:  UEI {peek.recipient_uei.notna().mean():.1%}"
      f"   DUNS {peek.recipient_duns.notna().mean():.1%}")

## The columns we keep

25 of 297, chosen from what the inspection above showed: identity of the contract,
who won it, who bought it, what was bought, how much, and when it runs out.
Reading all 297 exhausts a Colab runtime on the DoD file.

In [ ]:
KEEP_COLS = [
    "award_id_piid", "parent_award_id_piid", "modification_number", "action_date",
    "award_or_idv_flag", "award_type_code",
    "recipient_uei", "recipient_parent_uei", "recipient_parent_name", "recipient_name",
    "product_or_service_code", "product_or_service_code_description",
    "awarding_agency_name", "awarding_sub_agency_name",
    "awarding_office_code", "awarding_office_name",
    "federal_action_obligation", "current_total_value_of_award",
    "potential_total_value_of_award",
    # The end date of the predecessor is what later determines whether a follow-on
    # award is its successor rather than an unrelated buy from the same office.
    "period_of_performance_start_date", "period_of_performance_potential_end_date",
    "type_of_contract_pricing_code", "extent_competed_code",
    # FPDS reason FOC, "follow-on to competed action", is the only place the data
    # admits that one award succeeds another. Not a link to *which* one, but enough
    # to measure how many known follow-ons our matching actually finds.
    "other_than_full_and_open_competition_code",
    "naics_code",
]

print(f"keeping {len(KEEP_COLS)} of {peek.shape[1]} columns")
missing = [c for c in KEEP_COLS if c not in peek.columns]
assert not missing, f"columns not present in the archive: {missing}"

In [ ]:
# Each agency-year downloads independently, so one failure does not abort the rest.
# Failures are collected and reported; re-running this cell retries only what is
# still missing, since completed files are already cached.
targets = [(aid, label, fy) for fy in FISCAL_YEARS for aid, label in AGENCIES.items()]
failed = []
for agency_id, label, fy in targets:
    try:
        slim_one(agency_id, label, fy)
    except Exception as e:
        print(f"  !! {label} FY{fy}: {type(e).__name__}: {str(e)[:120]}", flush=True)
        failed.append(f"{label} FY{fy}")

cached = len(list(SLIM.glob("*.parquet")))
print(f"\ncache: {cached}/{len(targets)} agency-years")
if failed:
    print(f"{len(failed)} failed: {', '.join(failed)}")
    print("Re-run this cell - cached files are skipped, so only the failures are retried.")
else:
    print("cache complete")

# When restricted to a single fiscal year (GOVCON_FY set), the job is just building
# the cache - the analysis below runs separately, once, against the complete cache.
if os.environ.get("GOVCON_FY"):
    raise SystemExit(0)

## Assemble

`type_of_contract_pricing_code` and `extent_competed_code` hold the human-readable
description; the bare columns hold the letter codes - the opposite of what the
suffix implies, and not true of every column (see Check 1 above). And base-award
rows often obligate only a first increment, so `federal_action_obligation` badly
understates contract size - the potential value is the better size measure.

In [ ]:
AGENCY_SHORT = {
    "Department of Homeland Security": "DHS",
    "Department of Veterans Affairs": "VA",
    "General Services Administration": "GSA",
    "Department of Health and Human Services": "HHS",
}
KEEP_AGENCIES = ["Navy", "Army", "Air Force", "DHS", "VA", "GSA", "HHS"]
ANALYSIS_COLS = [
    "agency", "awarding_office_code", "awarding_office_name",
    "product_or_service_code", "product_or_service_code_description",
    "award_id_piid", "action_date", "pop_end", "value", "vendor",
    "recipient_parent_name", "pricing", "extent_competed_code", "naics_code",
    "population", "follow_on",
]

# What kind of contract an award is. These are three different questions wearing the
# same word: a standalone contract competed on its own, a vehicle like an IDIQ or
# GWAC being established, and an order placed under a vehicle someone already holds.
# Orders in particular go to the vehicle holder by construction, so pooling them
# would push retention toward 100% for reasons unrelated to winning anything.
def classify(d):
    parent = d["parent_award_id_piid"].fillna("").str.strip()
    return np.select(
        [d["award_or_idv_flag"].eq("IDV"),
         d["award_or_idv_flag"].eq("AWARD") & parent.eq("")],
        ["vehicle (IDV)", "standalone"],
        default="order under vehicle",
    )


def prepare(path):
    """Label, coalesce and filter one cached agency-year.

    Done per file rather than after one big concat on purpose. Seven years of raw
    string columns is roughly 3.8M rows x 23 object-dtype columns - several GB, and
    concat transiently doubles it, which exhausts a free Colab runtime. Filtering
    each file down to the analysis columns first cuts that by an order of magnitude.
    """
    d = pd.read_parquet(path)
    raw_n = len(d)

    # The military branches live inside the DoD file; everything else is its own agency.
    sub = d["awarding_sub_agency_name"].fillna("")
    top = d["awarding_agency_name"].fillna("")
    d["agency"] = np.where(sub.isin(DOD_SUBS), sub.map(DOD_SUBS), top.map(AGENCY_SHORT))
    d = d[d.agency.isin(KEEP_AGENCIES)]

    num = lambda c: pd.to_numeric(d[c], errors="coerce")
    # Base awards often obligate only a first increment, so the obligation alone
    # badly understates contract size - prefer the potential value.
    d["value"] = (num("potential_total_value_of_award")
                  .fillna(num("current_total_value_of_award"))
                  .fillna(num("federal_action_obligation")))
    d["action_date"] = pd.to_datetime(d["action_date"], errors="coerce")
    d["pop_end"] = pd.to_datetime(d["period_of_performance_potential_end_date"], errors="coerce")
    # Parent UEI, so a subsidiary re-winning its parent's work counts as retention.
    d["vendor"] = d["recipient_parent_uei"].fillna(d["recipient_uei"])
    d["recipient_parent_name"] = d["recipient_parent_name"].fillna(d["recipient_name"])
    d["pricing"] = d["type_of_contract_pricing_code"].fillna("")
    d["population"] = classify(d)
    # FPDS reason FOC, "follow-on to competed action" - the data's own admission that
    # one award succeeds another. This column holds the letter code, not the
    # description, unlike the two _code columns above.
    d["follow_on"] = d["other_than_full_and_open_competition_code"].fillna("").str.strip().eq("FOC")

    # Measured before the dropna, or it is trivially 100% and checks nothing.
    have_vendor, in_scope = int(d.vendor.notna().sum()), len(d)
    d = d.dropna(subset=["vendor", "action_date", "value"])
    d = d[d.value >= MIN_VALUE]
    return d[ANALYSIS_COLS], raw_n, len(d), have_vendor, in_scope


parts, raw_total, kept_total, vend_ok, vend_all = [], 0, 0, 0, 0
for path in sorted(SLIM.glob("*.parquet")):
    part, raw_n, kept_n, v_ok, v_all = prepare(path)
    parts.append(part)
    raw_total += raw_n
    vend_ok += v_ok
    vend_all += v_all

df_all = pd.concat(parts, ignore_index=True)
del parts
# Standalone contracts are the headline population; the others are compared below.
df = df_all[df_all.population == "standalone"].copy()
vendor_cov = vend_ok / max(vend_all, 1)

print(f"{raw_total:,} base awards read from {len(list(SLIM.glob('*.parquet')))} agency-years")
print(f"vendor key coverage: {vendor_cov:.2%}")
print(f"{len(df_all):,} awards >= ${MIN_VALUE:,} with a usable vendor and date")
print("\nby contract type:")
print(df_all.population.value_counts().to_string())
print(f"\nheadline population - standalone contracts: {len(df):,}")
print(df.groupby("agency").size().sort_values(ascending=False).to_string())

## What did we actually download?

Worth looking at the data before trusting anything computed from it. This section
is here so a reader can see the raw material rather than take the filtered numbers
on faith - what landed, what it looks like, and where the rows went.

In [ ]:
import matplotlib.pyplot as plt

# What is on disk, per agency-year.
inv = pd.DataFrame([
    {"file": f.stem, "rows": len(pd.read_parquet(f, columns=["award_id_piid"])),
     "MB": round(f.stat().st_size / 1e6, 1)}
    for f in sorted(SLIM.glob("*.parquet"))
])
print(f"{len(inv)} cached files, {inv.rows.sum():,} base awards, {inv.MB.sum():.0f} MB on disk\n")
print(inv.head(8).to_string(index=False))
print(f"... and {len(inv) - 8} more\n" if len(inv) > 8 else "")

# Actual awards, so the reader can see what a row is.
print("Sample of the awards being analysed:")
cols = ["agency", "action_date", "recipient_parent_name", "product_or_service_code", "value"]
print(df.sort_values("value", ascending=False)[cols].head(5).to_string(index=False), "\n")

In [ ]:
# Federal fiscal years start on 1 October, so October onward belongs to the next one.
df["fy"] = df.action_date.dt.year + (df.action_date.dt.month >= 10).astype(int)

print("Awards over $250K by agency and fiscal year:")
print(df.pivot_table(index="agency", columns="fy", values="award_id_piid",
                     aggfunc="count", fill_value=0).to_string(), "\n")

print("Contract value ($M):")
print((df.value / 1e6).describe(percentiles=[.5, .9, .99]).round(2).to_string(), "\n")

print("How the work was competed:")
print(df.extent_competed_code.fillna("(blank)").value_counts().head(6).to_string(), "\n")

print("Most common product/service codes:")
top = (df.groupby(["product_or_service_code", "product_or_service_code_description"])
         .size().sort_values(ascending=False).head(8))
print(top.to_string())

In [ ]:
# Where the rows went, filter by filter.
funnel = pd.DataFrame([
    ("Base awards downloaded (modifications already excluded)", raw_total),
    ("In the seven agencies studied", vend_all),
    (f"Usable vendor and date, over ${MIN_VALUE:,}", len(df)),
], columns=["step", "awards"])
funnel["share"] = (funnel.awards / raw_total).map(lambda v: f"{v:.1%}")
print(funnel.to_string(index=False))
print(f"\n{len(df) / raw_total:.1%} of base awards survive to the analysis")

fig, ax = plt.subplots(figsize=(9, 3.6), dpi=200)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
ax.hist(np.log10(df.value.clip(lower=1e4)), bins=60, color=BLUE, zorder=3)
ax.axvline(np.log10(MIN_VALUE), color=INK, lw=1.2, ls="--", zorder=4)
ax.text(np.log10(MIN_VALUE), ax.get_ylim()[1] * 0.92, f"  ${MIN_VALUE // 1000}K floor",
        fontsize=8.5, color=INK, va="top")
ax.set_xticks(range(4, 11), ["$10K", "$100K", "$1M", "$10M", "$100M", "$1B", "$10B"],
              fontsize=9, color=MUTED)
ax.set_xlabel("Contract value (log scale)", fontsize=9.5, color=MUTED)
ax.set_ylabel("Awards", fontsize=9.5, color=MUTED)
ax.tick_params(axis="y", labelsize=8, colors=MUTED, length=0)
ax.tick_params(axis="x", length=0)
ax.grid(axis="y", color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(False)
ax.set_title("Most federal contracts are small; a few are enormous",
             loc="left", fontsize=12, fontweight="bold", color=INK, pad=10)
fig.tight_layout()
fig.savefig(OUT / "value_distribution.png", facecolor=fig.get_facecolor())
plt.show()

## Build award events, then recompetes

A contracting office can award several contracts for the same product code on the
same day - multiple-award situations. Treating those as a sequence would
manufacture fake losses, so awards within 180 days collapse into one "event" whose
incumbent is a *set* of vendors. A later event counts as retained if it intersects
that set.

In [ ]:
def build_events(data, key, gap_days=EVENT_GAP_D):
    d = data.sort_values(key + ["action_date"])
    gap = d.groupby(key, sort=False)["action_date"].diff().dt.days
    d = d.assign(_new=gap.isna() | (gap > gap_days))
    d["event"] = d.groupby(key, sort=False)["_new"].cumsum()
    return (
        d.groupby(key + ["event"], sort=False)
        .agg(
            vendors=("vendor", lambda s: frozenset(s)),
            names=("recipient_parent_name", lambda s: "; ".join(sorted(set(s))[:3])),
            date=("action_date", "min"),
            end_date=("pop_end", "max"),
            value=("value", "max"),
            pricing=("pricing", "first"),
            competed=("extent_competed_code", "first"),
            follow_on=("follow_on", "any"),
            psc_desc=("product_or_service_code_description", "first"),
            office=("awarding_office_name", "first"),
            n_awards=("award_id_piid", "size"),
            piids=("award_id_piid", lambda s: "; ".join(sorted(set(s))[:3])),
        )
        .reset_index()
        # A stable id per event, so matching and the checks that read it back agree
        # on what an event is.
        .pipe(lambda d: d.assign(_id=np.arange(len(d))))
    )


def build_recompetes(events, key):
    """One row per consecutive pair of events on the same key.

    The loose reading of "the work came up again", kept as a sensitivity check.
    In a busy office the next award under a product code is usually a different
    requirement, so this over-counts recompetes and understates retention.
    """
    ev = events.sort_values(key + ["date"])
    g = ev.groupby(key, sort=False)
    out = ev.assign(
        prev_vendors=g["vendors"].shift(),
        prev_names=g["names"].shift(),
        prev_date=g["date"].shift(),
        prev_end=g["end_date"].shift(),
        prev_piids=g["piids"].shift(),
    )
    out = out[out.prev_vendors.notna()].copy()
    out["retained"] = [bool(a & b) for a, b in zip(out.vendors, out.prev_vendors)]
    out["gap_years"] = (out.date - out.prev_date).dt.days / 365.25
    return out


SUCC_COLS = ["_id", "date", "vendors", "names", "piids", "value", "pricing",
             "competed", "follow_on"]


def build_successors(events, key, window_d=LINK_WINDOW_D, unambiguous=True):
    """Match each expiring contract to the award that actually replaced it.

    Deliberately not "the next award in this group". One contracting office buys
    many unrelated things under a single product code, so the immediately following
    award is usually a different program. Pairing consecutively both invents losses
    between unrelated programs and discards the real successor, which sits three to
    five years out with other awards in between.

    A replacement announces itself by *timing*: it is awarded near the point the old
    contract runs out. So for each contract this searches the group for the award
    landing closest to its end date, within `window_d`.

    With `unambiguous`, a contract is skipped when more than one award sits in that
    window - there is no way to tell which one is the replacement, and guessing the
    nearest manufactures both false retentions and false losses. Recording nothing
    is the honest outcome: we cannot count what we cannot identify.
    """
    ev = events
    left = ev.dropna(subset=["end_date"]).sort_values("end_date")
    right = ev[key + SUCC_COLS].sort_values("date")
    tol = pd.Timedelta(days=window_d)

    def asof(direction):
        return pd.merge_asof(left, right, left_on="end_date", right_on="date", by=key,
                             direction=direction, tolerance=tol, suffixes=("_prev", ""))

    m = asof("nearest")
    if unambiguous:
        # The nearest award on each side of the end date are the only two candidates
        # that can be closest, so bracketing it settles ambiguity exactly.
        back, fwd = asof("backward"), asof("forward")
        b = back["_id"].where(back["_id"] != back["_id_prev"])
        f = fwd["_id"].where(fwd["_id"] != fwd["_id_prev"])
        n_cand = b.notna().astype(int) + f.notna().astype(int) - (b.eq(f) & b.notna()).astype(int)
        m = m[n_cand.to_numpy() <= 1]

    # merge_asof can return the contract itself when a short contract ends near its
    # own award date, so require a genuinely later, different award.
    m = m[m["_id"].notna() & (m["_id"] != m["_id_prev"]) & (m["date"] > m["date_prev"])].copy()

    # A replacement is roughly the size of the thing it replaces. An award a
    # thousand times larger is a different requirement that happened to land near
    # the old one's end date, and pairing them invents a loss.
    ratio = m["value"] / m["value_prev"]
    m = m[ratio.between(1 / VALUE_RATIO_MAX, VALUE_RATIO_MAX)].copy()

    m["retained"] = [bool(a & b) for a, b in zip(m["vendors"], m["vendors_prev"])]
    m["gap_years"] = (m["date"] - m["date_prev"]).dt.days / 365.25
    m = m.rename(columns={"names_prev": "prev_names", "piids_prev": "prev_piids",
                          "date_prev": "prev_date", "end_date": "prev_end"})
    return m


events = build_events(df, KEY)
linked = build_successors(events, KEY)      # headline: matched by contract end date
rec = build_recompetes(events, KEY)         # sensitivity: every consecutive pair
print(f"{len(events):,} award events")
print(f"{len(linked):,} successor-matched recompetes (headline)")
print(f"{len(rec):,} consecutive pairs (sensitivity)")

## Retention rate per agency

Reported with a Wilson 95% interval and the event count, because a bare percentage
with no n invites the "your sample is twelve contracts" objection.

In [ ]:
from statsmodels.stats.proportion import proportion_confint


def retention_table(recompetes, by="agency"):
    g = recompetes.groupby(by)["retained"]
    t = g.agg(n_recompetes="size", n_retained="sum").reset_index()
    t["retention_rate"] = t.n_retained / t.n_recompetes
    lo, hi = proportion_confint(t.n_retained, t.n_recompetes, method="wilson")
    t["ci_low"], t["ci_high"] = lo, hi
    return t.sort_values("retention_rate", ascending=False).reset_index(drop=True)


table = retention_table(linked)
overall = linked.retained.mean()
print(f"OVERALL: {overall:.1%} of recompetes stay with the incumbent (n={len(linked):,})\n")
print(table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
table.to_csv(OUT / "retention_by_agency.csv", index=False)

# Sensitivity: every consecutive pair, successor-linked or not. This is the looser
# reading of "the same work came up again" and it should sit lower, because it
# sweeps in unrelated buys from the same office.
loose = retention_table(rec)
print(f"\nSensitivity - all consecutive pairs, no successor linkage "
      f"(overall {rec.retained.mean():.1%}, n={len(rec):,}):")
print(loose.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
loose.to_csv(OUT / "retention_by_agency_unlinked.csv", index=False)

# "Does the incumbent win the recompete" presumes there was a competition. FPDS
# records whether there was, and a sole-source follow-on is a different event from
# a contest the incumbent had to win, so they are reported apart rather than pooled.
COMPETED = ("FULL AND OPEN COMPETITION", "FULL AND OPEN COMPETITION AFTER EXCLUSION OF SOURCES")
linked["was_competed"] = np.where(
    linked.competed.isin(COMPETED), "competed",
    np.where(linked.competed.fillna("") == "", "unknown", "not competed"))
comp = (linked.groupby("was_competed")["retained"]
              .agg(n="size", retention="mean").reset_index()
              .sort_values("n", ascending=False))
print("\nBy whether the follow-on was actually competed:")
print(comp.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
comp.to_csv(OUT / "retention_by_competition.csv", index=False)

competed_only = linked[linked.was_competed == "competed"]
if len(competed_only) >= MIN_N:
    print(f"\nCompeted follow-ons only - the strictest read of the question "
          f"({competed_only.retained.mean():.1%}, n={len(competed_only):,}):")
    print(retention_table(competed_only).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    retention_table(competed_only).to_csv(OUT / "retention_competed_only.csv", index=False)

## Does the answer depend on what counts as a contract?

The headline number covers standalone contracts only. An award can also be an IDIQ
or GWAC vehicle being established, or an order placed under a vehicle someone
already holds. This computes retention for those two populations too, so they can
be compared rather than left out silently.

In [ ]:
pop_rows = []
for pop in ["standalone", "vehicle (IDV)", "order under vehicle"]:
    sub = df_all[df_all.population == pop]
    if len(sub) < MIN_N:
        continue
    # Successor matching needs the prior contract's end date. A population where
    # that's rarely populated will show few or no recompetes for that reason alone,
    # so the coverage is reported alongside the count.
    has_end = sub.pop_end.notna().mean()
    r = build_successors(build_events(sub, KEY), KEY)
    if len(r) < MIN_N:
        print(f"  {pop}: {len(sub):,} awards, {len(r)} recompetes - not reported. "
              f"{has_end:.0%} of these awards carry a potential end date, which is "
              f"what the successor match needs.")
        continue
    pop_rows.append({"population": pop, "awards": len(sub), "recompetes": len(r),
                     "retention": r.retained.mean(), "has_end_date": has_end})
pops = pd.DataFrame(pop_rows)
print("Retention by what kind of contract is being recompeted:")
print(pops.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
pops.to_csv(OUT / "retention_by_population.csv", index=False)

## How much are we actually catching?

There is no ground truth for which award replaced which - the matching above is a
proxy. FPDS labels a subset of awards as a follow-on itself (reason code FOC),
which gives a small answer key to check the matching against.

In [ ]:
flagged = events[events.follow_on]
found = linked["_id"].astype("Int64")
recall = flagged["_id"].isin(set(found.dropna())).mean() if len(flagged) else float("nan")
print(f"{len(flagged):,} award events are labelled a follow-on by FPDS itself")
print(f"our matching links {recall:.1%} of them to a predecessor")
print("\nThis is a floor on recall, not a ceiling: FPDS only applies the label to "
      "\nsole-source follow-ons, so competed recompetes are never labelled and cannot "
      "\nbe checked this way.")

## Chart

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter



def retention_bar(t, title, subtitle, path):
    fig, ax = plt.subplots(figsize=(9, 5.2), dpi=200)
    fig.patch.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)

    x = np.arange(len(t))
    err = np.vstack([t.retention_rate - t.ci_low, t.ci_high - t.retention_rate])
    # A bar built on too few recompetes is drawn pale, so a thin result cannot be
    # read as carrying the same weight as the others.
    thin = t.n_recompetes < MIN_N
    ax.bar(x, t.retention_rate, width=0.62, color=np.where(thin, PALE, BLUE), zorder=3)
    ax.errorbar(x, t.retention_rate, yerr=err, fmt="none", ecolor=MUTED,
                elinewidth=1.2, capsize=4, zorder=4)

    for xi, row in zip(x, t.itertuples()):
        mark = "*" if row.n_recompetes < MIN_N else ""
        ax.text(xi, row.ci_high + 0.02, f"{row.retention_rate:.0%}{mark}",
                ha="center", va="bottom", fontsize=11, fontweight="bold", color=INK, zorder=5)
        ax.text(xi, -0.035, f"n={row.n_recompetes:,}",
                ha="center", va="top", fontsize=8, color=MUTED)

    ax.set_xticks(x, t.agency, fontsize=11, color=INK)
    ax.tick_params(axis="x", length=0, pad=16)
    ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.tick_params(axis="y", labelsize=9, colors=MUTED, length=0)
    ax.set_ylabel("Recompetes won by the incumbent", fontsize=10, color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)

    fig.suptitle(title, x=0.02, ha="left", fontsize=15, fontweight="bold", color=INK, y=0.98)
    ax.set_title(subtitle, loc="left", fontsize=9.5, color=MUTED, pad=14)
    fig.text(0.02, 0.01,
             f"Source: USASpending.gov Award Data Archive, {FY_LABEL}. Standalone contract awards "
             f"over ${MIN_VALUE//1000}K, grouped by contracting office, product code and NAICS,\nwhere the "
             "follow-on was awarded within a year of the prior contract's end date. "
             f"Bars show the 95% Wilson confidence interval.\n* fewer than {MIN_N} recompetes - "
             "too few to read as a firm result.",
             fontsize=7.5, color=MUTED, ha="left")
    fig.tight_layout(rect=[0, 0.05, 1, 0.96])
    fig.savefig(path, facecolor=fig.get_facecolor())
    return fig


# The title states the computed rate rather than a fixed claim, since it is written
# once the number is known.
retention_bar(
    table,
    f"Incumbents keep {overall:.0%} of federal recompetes",
    "Share of repeat contract awards that went back to the previous vendor, "
    f"{FY_LABEL} (n={len(linked):,})",
    OUT / "retention_by_agency.png",
)
plt.show()

## Stretch: does award size change the answer?

In [ ]:
bands = pd.cut(linked.value, [0, 5e6, 25e6, np.inf], labels=["<$5M", "$5-25M", ">$25M"])
size_tbl = (linked.assign(band=bands)
              .groupby(["agency", "band"], observed=True)["retained"]
              .agg(n="size", rate="mean").reset_index())
size_tbl = size_tbl[size_tbl.n >= 30]
print(size_tbl.pivot(index="agency", columns="band", values="rate").to_string(
    float_format=lambda v: f"{v:.1%}"))
size_tbl.to_csv(OUT / "retention_by_size.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5.2), dpi=200)
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
order = table.agency.tolist()
band_colors = {"<$5M": "#9ec5f4", "$5-25M": "#3987e5", ">$25M": "#184f95"}
width = 0.26
for i, (band, color) in enumerate(band_colors.items()):
    sub_t = size_tbl[size_tbl.band == band].set_index("agency").reindex(order)
    ax.bar(np.arange(len(order)) + (i - 1) * width, sub_t.rate.fillna(0),
           width=width - 0.02, color=color, label=band, zorder=3)
ax.set_xticks(np.arange(len(order)), order, fontsize=11, color=INK)
ax.tick_params(axis="x", length=0, pad=8)
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.tick_params(axis="y", labelsize=9, colors=MUTED, length=0)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncols=3, loc="upper right")
ax.set_ylabel("Recompetes won by the incumbent", fontsize=10, color=MUTED)
fig.suptitle("Retention by contract size", x=0.02, ha="left", fontsize=15, fontweight="bold", color=INK)
fig.tight_layout(rect=[0, 0.02, 1, 0.96])
fig.savefig(OUT / "retention_by_size.png", facecolor=fig.get_facecolor())
plt.show()

## Stretch: logistic regression

Which features actually move retention? Reported as odds ratios as well as raw
coefficients, because an odds ratio is the part a non-statistician can read.

In [ ]:
import statsmodels.formula.api as smf


def pricing_family(s):
    s = (s or "").upper()
    if "COST" in s:
        return "Cost Plus"
    if "FIXED PRICE" in s:
        return "Fixed Price"
    if "TIME AND MATERIALS" in s or "LABOR HOUR" in s:
        return "T&M / Labor Hours"
    return "Other"


model_df = linked.assign(
    family=linked.pricing.map(pricing_family),
    log_value=np.log10(linked.value.clip(lower=MIN_VALUE)),
    y=linked.retained.astype(int),
)
model_df = model_df[model_df.family != "Other"]

# Reference levels are read from the data rather than hardcoded, so the cell still
# runs if an agency drops out under a different filter.
ref_agency = model_df.agency.value_counts().idxmax()
ref_family = ("Fixed Price" if (model_df.family == "Fixed Price").any()
              else model_df.family.value_counts().idxmax())

fit = smf.logit(
    f'y ~ log_value + C(family, Treatment("{ref_family}")) + C(agency, Treatment("{ref_agency}"))',
    data=model_df,
).fit(disp=False)

# A logit that hit complete separation still returns coefficients - absurd ones,
# with odds ratios in the billions. Reporting those as findings would be worse
# than reporting nothing, so say so loudly instead of printing the table straight.
converged = fit.mle_retvals.get("converged", True)
if not converged:
    print("!! MODEL DID NOT CONVERGE (likely complete separation).")
    print("!! Coefficients and odds ratios below are not interpretable - do not report them.\n")

summary = pd.DataFrame({"coef": fit.params, "odds_ratio": np.exp(fit.params), "p_value": fit.pvalues})
print(f"reference levels: agency={ref_agency}, pricing={ref_family}")
print(fit.summary())
print("\nOdds ratios (>1 favours the incumbent):")
print(summary.to_string(float_format=lambda v: f"{v:.3f}"))
(OUT / "logit_summary.txt").write_text(
    ("" if converged else "MODEL DID NOT CONVERGE - coefficients are not interpretable.\n\n")
    + f"reference levels: agency={ref_agency}, pricing={ref_family}\n"
    + str(fit.summary()) + "\n\nOdds ratios (>1 favours the incumbent):\n" + summary.to_string()
)

## Verification

These are the checks that catch a wrong answer, not just a crash.

In [ ]:
# 1. type_of_contract_pricing_code should hold the description, not the letter code.
#    If this fails, every pricing feature in the regression is a meaningless letter.
assert df.pricing[df.pricing != ""].str.len().median() > 3, "pricing column looks like codes, not descriptions"

# 2. The vendor key must be near-complete, or we are silently comparing nothing.
assert vendor_cov > 0.99, f"vendor coverage only {vendor_cov:.2%}"

# 3. Enough evidence per agency to publish a bar. A warning, not an assertion:
#    whether a thin bar is publishable is a judgement call, and aborting here would
#    skip the checks below that are worth seeing either way.
thin = table[table.n_recompetes < MIN_N]
if not thin.empty:
    print(f"WARNING: under {MIN_N} recompetes, too noisy to publish - "
          f"{', '.join(f'{r.agency} (n={r.n_recompetes})' for r in thin.itertuples())}")
    print("Add fiscal years, or drop these agencies from the chart.")

# 4. Sensitivity ladder. Nothing in the data labels a recompete, so the number
#    depends on how "the same work came up again" is defined. Rather than defend
#    one definition, show what each choice is worth. If retention were still
#    climbing at the tightest grouping, the headline would be a lower bound rather
#    than an estimate - so this table is the honest read on how much to trust it.
LADDER = [
    ("agency + PSC (as briefed)", KEY_BRIEF),
    ("+ contracting office", ["agency", "awarding_office_code", "product_or_service_code"]),
    ("+ NAICS", KEY),
]
rows = []
for name, k in LADDER:
    ev_k = build_events(df, k)
    cons = build_recompetes(ev_k, k)
    amb = build_successors(ev_k, k, unambiguous=False)
    succ = build_successors(ev_k, k)
    rows.append({
        "grouping": name,
        "n_consecutive": len(cons), "consecutive": cons.retained.mean(),
        "n_nearest": len(amb), "nearest": amb.retained.mean(),
        "n_unambiguous": len(succ), "unambiguous": succ.retained.mean(),
    })
ladder = pd.DataFrame(rows)
print("Sensitivity ladder - how the definition of a recompete moves the answer:")
print(ladder.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
ladder.to_csv(OUT / "sensitivity_ladder.csv", index=False)

brief_tbl = retention_table(build_successors(build_events(df, KEY_BRIEF), KEY_BRIEF))
brief_tbl.to_csv(OUT / "retention_by_agency_psc_only.csv", index=False)

# 5. Eyeball a real sequence. Confirm the retained/lost labels by hand,
#    and cross-check a PIID on usaspending.gov.
spot_key = linked.sort_values("value", ascending=False).iloc[0][KEY].tolist()
mask = np.logical_and.reduce([events[k] == v for k, v in zip(KEY, spot_key)])
print("\nSpot check -", spot_key)
print(events[mask].sort_values("date")[
    ["date", "names", "value", "n_awards", "piids", "psc_desc"]
].to_string(index=False))

# 6. Distribution of gaps between awards - should look like real contract lengths,
#    clustering in the 1-5 year range rather than piling up at the boundary.
print("\nYears between consecutive awards:")
print(linked.gap_years.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_string())